
# Task 3 — Image Quality Metrics (Production QC)

This notebook implements **two complementary image-quality metrics** for the provided production dataset:

1. **SNR** (signal-to-noise ratio): detects low-signal / noisy acquisitions.
2. **Sharpness** via **variance of the 3D Laplacian**: detects blur/motion and loss of high-frequency content.

Together, these separate images into **good** and **bad** quality groups and provide **threshold guidance** for deployment.



## Dataset
- Input file: `Task3/Images/images.nii.gz` (provided here as `images.nii`)
- The loaded array has shape **(X, Y, Z, N)** where **N is the number of production images**.
- Indexing in outputs below is **0-based** (Python convention). If needed, convert to 1-based by adding 1.



## Reproducibility notes
To reduce run-to-run differences and noisy console output:
- We set stable printing/formatting (rounded to a few decimals).
- We avoid printing large arrays unless explicitly needed.
- Metric computation is deterministic (no randomness).


In [ ]:

import os, struct, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Display/printing controls (keeps output stable & readable) ----------
pd.set_option("display.precision", 4)
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)
np.set_printoptions(precision=4, suppress=True)

def fmt(x, nd=4):
    """Consistent float formatting for prints."""
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

print("Environment versions:")
print("  Python      :", sys.version.split()[0])
print("  Platform    :", platform.platform())
print("  NumPy       :", np.__version__)
print("  Pandas      :", pd.__version__)
print("  Matplotlib  :", plt.matplotlib.__version__)



## Minimal NIfTI-1 `.nii` loader (no external dependencies)
This reads the header, then memory-maps the data payload. It supports common datatypes and applies NIfTI scaling.


In [ ]:

def load_nifti_nii(path):
    with open(path, "rb") as f:
        hdr = f.read(348)

    sizeof_hdr = struct.unpack("<i", hdr[0:4])[0]
    if sizeof_hdr != 348:
        sizeof_hdr = struct.unpack(">i", hdr[0:4])[0]
        endian = ">"
        if sizeof_hdr != 348:
            raise ValueError("Not a NIfTI-1 .nii file (unexpected header size).")
    else:
        endian = "<"

    dim = struct.unpack(endian + "8h", hdr[40:40+16])
    ndim = dim[0]
    shape = tuple(int(d) for d in dim[1:ndim+1])

    datatype = struct.unpack(endian + "h", hdr[70:72])[0]
    vox_offset = struct.unpack(endian + "f", hdr[108:112])[0]
    scl_slope = struct.unpack(endian + "f", hdr[112:116])[0]
    scl_inter = struct.unpack(endian + "f", hdr[116:120])[0]

    dt_map = {
        2: np.uint8, 4: np.int16, 8: np.int32, 16: np.float32, 64: np.float64,
        256: np.int8, 512: np.uint16, 768: np.uint32, 1024: np.int64, 1280: np.uint64,
    }
    if datatype not in dt_map:
        raise ValueError(f"Unsupported NIfTI datatype code: {datatype}")

    np_dtype = dt_map[datatype]

    mm = np.memmap(path, dtype=np_dtype, mode="r", offset=int(vox_offset))
    nvox = int(np.prod(shape))
    data = np.array(mm[:nvox]).reshape(shape, order="F").astype(np.float32)

    # Apply scaling (as per NIfTI spec)
    if scl_slope == 0.0:
        scl_slope = 1.0
    if scl_slope != 1.0 or scl_inter != 0.0:
        data = data * float(scl_slope) + float(scl_inter)

    meta = {
        "shape": shape,
        "datatype": datatype,
        "vox_offset": float(vox_offset),
        "scl_slope": float(scl_slope),
        "scl_inter": float(scl_inter),
        "endian": endian,
    }
    return data, meta



## Load data
Update `path` if you renamed or moved the `.nii` file.


In [ ]:

# Path (edit if you move the file)
path = "images.nii"  # update if needed

if not os.path.exists(path):
    raise FileNotFoundError(f"Could not find {path}. Place it next to this notebook or update `path`.")

data, meta = load_nifti_nii(path)

print("Meta:", meta)
print("Data shape:", data.shape, "dtype:", data.dtype)
print("Min/Max:", fmt(data.min(), 4), "/", fmt(data.max(), 4))



## Metric computation
### Foreground mask + SNR
- Foreground mask uses a robust threshold based on high percentiles.
- Noise estimate uses the darkest 20% of background voxels (fallback to all background if too few).

### Sharpness: 3D Laplacian variance
- Uses a 6-neighbour discrete Laplacian (fast, dependency-free).
- Sharpness is computed as variance of the Laplacian values inside the foreground.


In [ ]:

def compute_metrics(vol):
    # Foreground mask via robust thresholding
    vol = np.asarray(vol, dtype=np.float32)
    flat = vol.reshape(-1)

    p98 = np.percentile(flat, 98)
    thr = 0.1 * p98
    fg = vol > thr
    if fg.sum() < 100:
        thr = np.percentile(flat, 85)
        fg = vol > thr

    bg = vol[~fg]
    if bg.size == 0:
        bg = flat

    # Noise from darkest 20% of background
    p20 = np.percentile(bg, 20)
    noise = bg[bg <= p20]
    if noise.size < 50:
        noise = bg

    sigma = float(np.std(noise, ddof=1) + 1e-6)  # ddof=1 for sample std; +eps for safety
    mu = float(np.mean(vol[fg]) if fg.sum() else np.mean(vol))
    snr = mu / sigma

    # 3D 6-neighbour Laplacian
    I = vol
    lap = (-6.0 * I
           + np.roll(I, 1, axis=0) + np.roll(I, -1, axis=0)
           + np.roll(I, 1, axis=1) + np.roll(I, -1, axis=1)
           + np.roll(I, 1, axis=2) + np.roll(I, -1, axis=2))

    lap_fg = lap[fg] if fg.sum() else lap.reshape(-1)
    sharp = float(np.var(lap_fg))

    return float(snr), float(sharp)



## Compute per-image metrics
We assume the dataset is **(X, Y, Z, N)**. If you have a single volume, the code still works.


In [ ]:

n_imgs = data.shape[3] if data.ndim == 4 else 1
snrs, sharps = [], []

for i in range(n_imgs):
    vol = data[..., i] if data.ndim == 4 else data
    s, sh = compute_metrics(vol)
    snrs.append(s)
    sharps.append(sh)

df = pd.DataFrame({
    "index": np.arange(n_imgs, dtype=int),
    "snr": np.array(snrs, dtype=np.float64),
    "sharpness_var_lap": np.array(sharps, dtype=np.float64),
})

# Show summary without excessive decimals
df.describe().round(4)



## Visualisation (sanity checks)


In [ ]:

plt.figure()
plt.scatter(df["snr"], df["sharpness_var_lap"])
plt.xlabel("SNR (mean foreground / std noise)")
plt.ylabel("Sharpness (Var of 3D Laplacian)")
plt.title("QC metrics per image")
plt.show()

plt.figure()
plt.hist(df["snr"], bins=20)
plt.xlabel("SNR")
plt.ylabel("Count")
plt.title("SNR distribution")
plt.show()

plt.figure()
plt.hist(df["sharpness_var_lap"], bins=20)
plt.xlabel("Sharpness (Var Laplacian)")
plt.ylabel("Count")
plt.title("Sharpness distribution")
plt.show()



## Thresholding strategy (deployment guidance)

Production datasets often fail via two dominant modes:
- **Low SNR** (noisy / low-signal scans, gain issues, drift)
- **Low sharpness** (motion blur, conveyor synchronisation, gradient issues)

A robust and interpretable deployment rule is:

\[
\textbf{Good} \iff SNR \ge T_{SNR} \;\wedge\; Sharpness \ge T_{Sharp}
\]

Here we set thresholds using the **15th percentile** of the dataset as a conservative starting point.
In production, tune these thresholds using labeled QC outcomes or by targeting a desired reject rate.


In [ ]:

# Use float64 for percentile stability; format prints to a fixed number of decimals
snr_thr = float(np.percentile(df["snr"].values.astype(np.float64), 15))
sharp_thr = float(np.percentile(df["sharpness_var_lap"].values.astype(np.float64), 15))

df["is_good"] = (df["snr"] >= snr_thr) & (df["sharpness_var_lap"] >= sharp_thr)

good_idx = df.loc[df["is_good"], "index"].astype(int).tolist()
bad_idx  = df.loc[~df["is_good"], "index"].astype(int).tolist()

print("Thresholds (rounded for display):")
print("  T_SNR   =", fmt(snr_thr, 4))
print("  T_Sharp =", fmt(sharp_thr, 4))
print()
print(f"Good images: {len(good_idx)} / {len(df)}")
print(f"Bad images : {len(bad_idx)} / {len(df)}")


In [ ]:

plt.figure()
plt.scatter(df["snr"], df["sharpness_var_lap"], c=df["is_good"].astype(int))
plt.axvline(snr_thr, linestyle="--")
plt.axhline(sharp_thr, linestyle="--")
plt.xlabel("SNR")
plt.ylabel("Sharpness (Var Laplacian)")
plt.title("Good/Bad classification using two-metric thresholds")
plt.show()



## Required outputs
- Indexes of good-quality images
- Indexes of bad-quality images

(0-based indexing)

To keep notebook output clean, we print the **counts** and the first few indices, and save the full lists to disk.


In [ ]:

print("GOOD indexes (0-based):")
print(good_idx[:20], ("..." if len(good_idx) > 20 else ""))
print()
print("BAD indexes (0-based):")
print(bad_idx[:20], ("..." if len(bad_idx) > 20 else ""))

# Save full outputs
df.to_csv("task3_qc_metrics_with_labels.csv", index=False)

with open("task3_good_indices.txt", "w") as f:
    f.write(",".join(map(str, good_idx)))

with open("task3_bad_indices.txt", "w") as f:
    f.write(",".join(map(str, bad_idx)))

print("\nSaved:")
print("  task3_qc_metrics_with_labels.csv")
print("  task3_good_indices.txt")
print("  task3_bad_indices.txt")



## Notes on run-to-run stability
- Metric calculation is deterministic (no RNG).
- Percentiles are computed using float64 for stable thresholds.
- Printed values are rounded for readability; saved CSV contains full precision.
